In [18]:
import time
from pathlib import Path

import requests

from src.core import *
from src.gpt import *

In [19]:
np.random.seed(42)

In [20]:
class GPTEmbeddingWithCache(GPTEmbedding):

    def forward(self, x: Tensor, offset=0):
        token = self.embedding(x)
        position = self.positional_embedding(Tensor(range(offset, offset + x.shape[1])))
        return self.dropout(token + position)

In [21]:
class GPTAttentionWithCache(GPTAttention):

    def forward(self, x: Tensor, use_cache=False, cache=None):
        norm = self.normalize(x)
        head_embedding_size = self.embedding_size // self.heads
        multi_head_shape = (-1, x.shape[1], self.heads, head_embedding_size)
        shape = (-1, x.shape[1], self.embedding_size)

        query = self.query(norm).reshape(multi_head_shape).transpose((0, 2, 1, 3))
        key = self.key(norm).reshape(multi_head_shape).transpose((0, 2, 3, 1))
        value = self.value(norm).reshape(multi_head_shape).transpose((0, 2, 1, 3))

        if use_cache and cache is not None:
            key = Tensor(np.concatenate([cache[0].data, key.data], axis=3))
            value = Tensor(np.concatenate([cache[1].data, value.data], axis=2))
        new_cache = (key, value) if use_cache else None

        scale = Tensor(np.array(1.0 / np.sqrt(head_embedding_size)))
        scores = query @ key * scale
        if cache is None:
            scores = self.mask(scores)
        weights = self.softmax(scores)
        output = self.output((weights @ value).transpose((0, 2, 1, 3)).reshape(shape))
        return x + self.dropout(output), new_cache


In [22]:
class GPTTransformerWithCache(Composite):

    def __init__(self, embedding_size, heads):
        self.attention = GPTAttentionWithCache(embedding_size, heads)
        self.feed_forward = GPTFeedForward(embedding_size)

        super().__init__([self.attention,
                          self.feed_forward])

    def forward(self, x: Tensor, use_cache=False, cache=None):
        x, new_cache = self.attention(x, use_cache, cache)
        return self.feed_forward(x), new_cache

In [23]:
class GPTWithCache(Composite):

    def __init__(self, vocab_size, context_size, embedding_size, heads, blocks):
        self.embedding = GPTEmbeddingWithCache(vocab_size, context_size, embedding_size)
        self.transformers = [GPTTransformerWithCache(embedding_size, heads) for _ in range(blocks)]
        self.output = GPTOutput(embedding_size, vocab_size)

        super().__init__([self.embedding] + self.transformers + [self.output])

    def forward(self, x: Tensor, use_cache=False, caches=None, offset=0):
        x = self.embedding(x, offset)
        new_caches = []
        for i, layer in enumerate(self.transformers):
            cache = caches[i] if caches is not None else None
            x, new_cache = layer(x, use_cache, cache)
            new_caches.append(new_cache)
        return self.output(x), new_caches

In [24]:
class GPTTrainerWithCache(GPTTrainer):

    def generate(self, dataset, prompt, steps=256, use_cache=True):
        self.model.eval()
        tokens = dataset.encode(prompt)
        caches = None

        with Tensor.no_grad():
            for _ in range(steps):
                if use_cache and caches is not None and len(tokens) <= dataset.context_size:
                    feature = Tensor([tokens[-1:]])
                    offset = len(tokens) - 1
                else:
                    feature = Tensor([tokens[-dataset.context_size:]])
                    offset = 0
                    caches = None

                logits, caches = self.model(feature, use_cache, caches, offset)

                last_logits = logits.data[0, -1]
                exp = np.exp(last_logits - np.max(last_logits))
                probs = exp / np.sum(exp)
                token = np.random.choice(len(probs), p=probs)
                tokens.append(token)

        return dataset.decode(tokens)

In [25]:
LEARNING_RATE = 0.001

In [26]:
BATCH_SIZE = 16

In [27]:
CONTEXT_SIZE = 256

In [28]:
EMBEDDING_SIZE = 384

In [29]:
HEADS = 6

In [30]:
BLOCKS = 6

In [31]:
DATA_FILE = "../../../tinyshakespeare.txt"

file = Path(DATA_FILE)
if not file.exists():
    file.parent.mkdir(parents=True, exist_ok=True)
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    response = requests.get(url)
    response.raise_for_status()
    file.write_text(response.text)

In [32]:
MODEL_FILE = "../../../tinyshakespeare-gpt.npz"

dataset = CharDataset(DATA_FILE, BATCH_SIZE, CONTEXT_SIZE)
model = GPTWithCache(dataset.vocab_size, CONTEXT_SIZE, EMBEDDING_SIZE, HEADS, BLOCKS)
loss_fn = GPTLoss()
optimizer = AdamWOptimizer(model.parameters, lr=LEARNING_RATE)
trainer = GPTTrainerWithCache(model, loss_fn, optimizer)
trainer.load(MODEL_FILE)

In [33]:
np.random.seed(42)

start = time.time()
print(trainer.generate(dataset, prompt="ROMEO:", use_cache=False))
print(f"{time.time() - start:.8f} seconds")

ROMEO:
When I no woman me, I have negless'd thee
Which customs can do him:
You sir.

BUCKINGHAM:
There in English be, whose thou begin
Of abused wonder than he was done.

QUEEN ELIZABETH:
Wilt thou wilt to salt Cap, in direction
Making the dom office whose his l
4.05393887 seconds


In [34]:
np.random.seed(42)

start = time.time()
print(trainer.generate(dataset, prompt="ROMEO:", use_cache=True))
print(f"{time.time() - start:.8f} seconds")

ROMEO:
When I no woman me, I have negless'd thee
Which customs can do him:
You sir.

BUCKINGHAM:
There in English be, whose thou begin
Of abused wonder than he was done.

QUEEN ELIZABETH:
Wilt thou wilt to salt Cap, in direction
Making the dom office whose his l
0.52479720 seconds
